In [1]:
# Less Salt Less Sugar Monthly Report (launch from 31Aug2026. last for 1 year at least)
#     1. Daily Badge impression by device 
#     2. Daily Brand Page (LMS) Pageview/ Impressions   
#     3. Daily Theme Listing Impressions (adv search)

#http://192.168.61.59:8888/notebooks/reports/monthly_icon_impression_report/monthly_icon_impression_report_result.ipynb

# $: 142.24x3=426.72GB
# 1TB $50, 426.72GB ≈ $21.34

In [4]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [55]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month

all_dates = pd.date_range(
    start=f"{year}-{str_month}-01",
    end=pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
    freq="D",
)

str_month

'09'

In [56]:
template_file = 'Pet_report_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

# excel_name = "Pet_report_2026-10-05.xlsx"

'Pet_report_2026-10-09.xlsx'

In [6]:
#  1. Quick Search
sql_1 = f"""
SELECT
    DATE(time) AS querydate, platform, COUNT(1) AS count
FROM `openrice-production.ORGA.PV_{year}{str_month}*`
WHERE
    (
        LOWER(platform) IN ('desktop', 'mobile')        -- Web
        AND LOWER(EventAction) IN ('view.sr1.promotion')
        AND LOWER(EventLabelRaw) LIKE '%promotionid:100%amenityid=20232%'
    )
    OR
    (
        LOWER(platform) IN ('android', 'ios', 'hms')        -- App
        AND LOWER(EventAction) IN ('or.search.quick')
        AND LOWER(EventLabelRaw) LIKE '%amtid:20232%sn:hk.qcksearch%'  
    )
GROUP BY querydate, platform
"""

df_bq_1 = client.query(sql_1).result().to_dataframe()

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [57]:
pivoted_df_1 = df_bq_1.pivot(index='querydate', columns='platform', values='count')
pivoted_df_1 = pivoted_df_1.reindex(index=all_dates, fill_value=0).fillna(0)

expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_1 = pivoted_df_1.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_1["Web"]=pivoted_df_1["desktop"] + pivoted_df_1["mobile"]
pivoted_df_1["App"]=pivoted_df_1["ios"] + pivoted_df_1["android"] + pivoted_df_1["hms"]
pivoted_df_1 =pivoted_df_1[["Web","App"]]

pivoted_df_1.index.name = None
pivoted_df_1.reset_index(inplace=True)
pivoted_df_1["index"] = pivoted_df_1["index"].dt.date
pivoted_df_1

platform,index,Web,App
0,2026-09-01,15,13
1,2026-09-02,13,11
2,2026-09-03,16,26
3,2026-09-04,34,22
4,2026-09-05,5,25
5,2026-09-06,15,29
6,2026-09-07,17,5
7,2026-09-08,28,74
8,2026-09-09,29,31
9,2026-09-10,18,19


In [58]:
# group by platform but not date
df_bq_1_platform = df_bq_1.groupby('platform', as_index=False)['count'].sum()
df_bq_1_platform

,platform,count
0,android,264
1,desktop,530
2,hms,7
3,ios,1997
4,mobile,20


In [ ]:
#  1.1 Quick Search (another version)

# EventLabel example:
# "{'list': array([{'item': {'List': 0, 'Param': 'Page', 'Value': '1'}},
#  {'item': {'List': 0, 'Param': 'whatwhere', 'Value': '串燒兄妹'}},
#  {'item': {'List': 0, 'Param': 'savehistory', 'Value': 'true'}},
#  {'item': {'List': 0, 'Param': 'CityID', 'Value': '0'}},
#  {'item': {'List': 0, 'Param': 'Lang', 'Value': 'hk'}},
#  {'item': {'List': 0, 'Param': 'Ver', 'Value': '7.20.5'}},
#  {'item': {'List': 0, 'Param': 'sr', 'Value': 'qcksearch'}},
#  {'item': {'List': 0, 'Param': 'sn', 'Value': 'HK.Search.SR1.1'}}], dtype=object)}"

sql_1_1 = r"""

      SELECT
        LOWER(Platform) AS platform,
        COUNT(1) AS count
      FROM `openrice-production.ORGA.PV_20260930`
      WHERE 
      (
        LOWER(Platform) IN ('android', 'ios', 'hms')
        AND LOWER(EventAction) = 'or.search.quick'
        AND EXISTS (
        SELECT 1 FROM UNNEST(EventLabel.list) AS e
        WHERE LOWER(e.item.Param) = 'sn' AND LOWER(CAST(e.item.Value AS STRING)) = 'hk.qcksearch'
        )
        AND EXISTS (
        SELECT 1 FROM UNNEST(EventLabel.list) AS e
        WHERE LOWER(e.item.Param) = 'amtid' AND CAST(e.item.Value AS STRING) = '20232'
        )
      )
      OR
      (
        LOWER(Platform) IN ('desktop', 'mobile')
        AND LOWER(EventAction) = 'view.sr1.promotion'
        AND EXISTS (
        SELECT 1 FROM UNNEST(EventLabel.list) AS e
        WHERE LOWER(e.item.Param) = 'promotionid' AND CAST(e.item.Value AS STRING) = '100'
        )
        AND EXISTS (
        SELECT 1 FROM UNNEST(EventLabel.list) AS e
        WHERE LOWER(e.item.Param) = 'sn' AND LOWER(CAST(e.item.Value AS STRING)) LIKE '%amenityid=20232%'
        )
      )
  
    GROUP BY platform
    ORDER BY platform
"""

df_1_1 = client.query(sql_1_1).result().to_dataframe()
df_1_1

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,platform,count
0,android,26
1,desktop,9
2,ios,250
3,mobile,3


In [60]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df_1.to_excel(writer, sheet_name='Sheet1', startrow=2, header=None, index=False)

In [61]:
# 2. Adv Search

sql_2 = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE    
        (   
            LOWER(platform) IN ('desktop', 'mobile')           -- web
            and LOWER(EventAction) = 'or.search.adv'
            and LOWER(EventLabelRaw) LIKE '%conditionid:11024%amenityid=20232%'
        )  
        OR   
        (   LOWER(platform) in ('android','ios','hms')         -- app
            and LOWER(EventAction) = 'or.search.adv'
            and LOWER(EventLabelRaw) LIKE '%featureflt:11024%sn:hk.advsearch%'
        )  

    group by platform, querydate
    """

df_bq_2 = client.query(sql_2).result().to_dataframe()

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
#df_bq_2

In [62]:
pivoted_df_2 = df_bq_2.pivot(index='querydate', columns='platform', values='count')
pivoted_df_2 = pivoted_df_2.reindex(index=all_dates, fill_value=0).fillna(0)
expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_2 = pivoted_df_2.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_2["Web"]= pivoted_df_2["desktop"] + pivoted_df_2["mobile"]
pivoted_df_2["App"]=pivoted_df_2["ios"] + pivoted_df_2["android"] + pivoted_df_2["hms"]
pivoted_df_2 =pivoted_df_2[["Web","App"]]

pivoted_df_2.index.name = None
pivoted_df_2.reset_index(inplace=True)
pivoted_df_2["index"] = pivoted_df_2["index"].dt.date
pivoted_df_2

platform,index,Web,App
0,2026-09-01,0,51
1,2026-09-02,0,58
2,2026-09-03,0,82
3,2026-09-04,1,136
4,2026-09-05,0,286
5,2026-09-06,0,216
6,2026-09-07,0,103
7,2026-09-08,0,87
8,2026-09-09,0,116
9,2026-09-10,0,110


In [63]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df_2.to_excel(writer, sheet_name='Sheet1', startrow=37, header=None, index=False)

Below is Testing!!!

is Testing!!!

Testing!!!

In [53]:
# adv search
# and STRPOS(LOWER(COALESCE(EventLabelRaw, '')),'promotionid:100') > 0 --> contain %

sql_test =f"""

    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_20261008`
    WHERE    
        (   
            LOWER(platform) IN ('desktop', 'mobile')           -- web
            and LOWER(EventAction) = 'or.search.adv'
            and LOWER(EventLabelRaw) LIKE '%conditionid:11024%amenityid=20232%'
        )            --%conditionid:11024
        OR   
        (   LOWER(platform) in ('android','ios','hms')         -- app
            and LOWER(EventAction) = 'or.search.adv'
            and LOWER(EventLabelRaw) LIKE '%featureflt:11024%'
        )  
-- sn:hk.advsearch, featureflt:11024
    group by platform, querydate
    """
df_bq_test = client.query(sql_test).result().to_dataframe()
df_bq_test


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,querydate,platform,count
0,2026-10-08,ios,78
1,2026-10-08,android,16


In [48]:
expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_test = (
    df_bq_test
    .pivot(index="querydate", columns="platform", values="count")
    .reindex(columns=expected_platforms, fill_value=0)
    .fillna(0)
)
pivoted_df_test["Web"] = (pivoted_df_test["desktop"] + pivoted_df_test["mobile"])
pivoted_df_test["App"] = (pivoted_df_test["ios"]+ pivoted_df_test["android"]+ pivoted_df_test["hms"])

pivoted_df_test = pivoted_df_test[["Web", "App"]].reset_index()
pivoted_df_test["querydate"] = pd.to_datetime(pivoted_df_test["querydate"]).dt.date

pivoted_df_test

platform,querydate,Web,App
0,2026-10-08,0,94


In [54]:
#  2.1 Adv Search

# EventLabel example:
# "{'list': array([{'item': {'List': 0, 'Param': 'Page', 'Value': '1'}},
#  {'item': {'List': 0, 'Param': 'whatwhere', 'Value': '串燒兄妹'}},
#  {'item': {'List': 0, 'Param': 'savehistory', 'Value': 'true'}},
#  {'item': {'List': 0, 'Param': 'CityID', 'Value': '0'}},
#  {'item': {'List': 0, 'Param': 'Lang', 'Value': 'hk'}},
#  {'item': {'List': 0, 'Param': 'Ver', 'Value': '7.20.5'}},
#  {'item': {'List': 0, 'Param': 'sr', 'Value': 'qcksearch'}},
#  {'item': {'List': 0, 'Param': 'sn', 'Value': 'HK.Search.SR1.1'}}], dtype=object)}"

sql_2_1 = r"""

      SELECT
        LOWER(Platform) AS platform,
        COUNT(1) AS count
      FROM `openrice-production.ORGA.PV_20261008`
      WHERE LOWER(EventAction) = 'or.search.adv'
      AND  (
      (
        LOWER(Platform) IN ('android', 'ios', 'hms')
        AND EXISTS (
        SELECT 1 FROM UNNEST(EventLabel.list) AS e
        WHERE LOWER(e.item.Param) = 'sn' AND LOWER(CAST(e.item.Value AS STRING)) = 'hk.advsearch'
        )
        AND EXISTS (
        SELECT 1 FROM UNNEST(EventLabel.list) AS e
        WHERE LOWER(e.item.Param) = 'featureflt' AND CAST(e.item.Value AS STRING) = '11024'
        )
      )
      OR
      (
        LOWER(Platform) IN ('desktop', 'mobile')
        AND EXISTS (
        SELECT 1 FROM UNNEST(EventLabel.list) AS e
        WHERE LOWER(e.item.Param) = 'conditionid' AND CAST(e.item.Value AS STRING) = '11024'
        )
        AND EXISTS (
        SELECT 1 FROM UNNEST(EventLabel.list) AS e
        WHERE LOWER(e.item.Param) = 'sn' AND LOWER(CAST(e.item.Value AS STRING)) LIKE '%amenityid=20232%'
        )
      )
    )
    GROUP BY platform
    ORDER BY platform
"""

df_2_1 = client.query(sql_2_1).result().to_dataframe()
df_2_1

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,platform,count
0,android,14
1,ios,71


In [ ]:
# slack record
#1. Quick Search
#   - Web: view.SR1.Promotion | CityID:0;PromotionId:100;Sn:https://www.openrice.com/zh/hongkong/restaurants?amenityId=20232&tabIndex=0
#   - App: android: 11:53:37|| or.search.quick| CityID:0;AmtID:20232;Page:1;Lang:zh_TW;Ver:7.21.0; sn:hk.QckSearch
#          ios: 17:12:45|| or.search.quick| CityID:0;Lang:hk;Ver:7.21.0;AmtID:20232;sn:HK.QckSearch

#2. Adv Search
#   - Web: or.search.adv | CityID:0;Sr:Biz;SortBy:ORScoreDesc;ConditionId:11024;UiCity:hongkong;Sn:https://www.openrice.com/zh/hongkong/restaurants?amenityId=20232&tabIndex=0
#   - App: android: 11:54:44|| or.search.adv| CityID:0;SortBy:orscoredesc;Page:1;FeatureFlt:11024;Lang:zh_TW;Ver:7.21.0; sn:hk.AdvSearch
#          ios: 17:13:40|| or.search.adv| FeatureFlt:11024;Page:1;savehistory:true;CityID:0;Lang:hk;Ver:7.21.0;sr:(null);sn:HK.AdvSearch